In [1]:
import sys
sys.path.append('/home/ronedr/evolution-strategy-baselines-comparison')

In [2]:
## imports.
import jax
import optax
from tqdm import tqdm
import brax.envs as brax_envs
from evosax.problems import BraxProblem as Problem
from evosax.problems.networks import MLP, tanh_output_fn
from experiment.utils.problem_utils import get_problem_settings
from experiment.run_experiments import run_experiment_permutations
from evosax.core.fitness_shaping import standardize_fitness_shaping_fn

In [3]:
# running params.
num_generations = 1000
population_size = 256
eval_batch_size = None
log_period = 1
result_dir = "../../results"
problems_brax_envs = list(brax_envs._envs.keys())

In [4]:
import optax
lr_schedule = optax.exponential_decay(
    init_value=0.01,
    transition_steps=num_generations,
    decay_rate=0.1,
)
std_schedule = optax.exponential_decay(
    init_value=0.05,
    transition_steps=num_generations,
    decay_rate=0.2,
)

# all algorithms that we want to comapre with best params according the article "DISCOVERING EVOLUTION STRATEGIES VIA META-BLACK-BOX OPTIMIZATION".
es_dict = {
    "Open_ES": {    
        "optimizer": optax.adam(learning_rate=lr_schedule),
        "std_schedule": std_schedule,
        # "optimizer": optax.adam(learning_rate=0.05)
    },
    # "PGPE": {
    #     "optimizer": optax.adam(learning_rate=0.02),
    # },
    # "ASEBO": {
    #     "optimizer": optax.adam(learning_rate=0.01),
    #     "fitness_shaping_fn": standardize_fitness_shaping_fn
    # },
    # "LES": {},
    # "SNES": {},
    # "Sep_CMA_ES": {},
    # "CMA_ES": {},
    # "DES": {},
    # "EvoTF_ES": {},
}
running_es = es_dict



In [5]:
for env_name in tqdm(problems_brax_envs, desc="Loading Problems .."):
    action_num, out_fn = get_problem_settings(env_name)
    try:
        problem = Problem(
            num_rollouts=1,
            env_name=env_name,
            policy=MLP(
                layer_sizes=(32, 32, 32, 32, action_num),
                output_fn=out_fn,
            ),
            episode_length=1000
        )
        print("Successfully loaded:", env_name)
        run_experiment_permutations(problems=[problem],
                                    es_dict=running_es,
                                    num_generations=num_generations,
                                    population_size=population_size,
                                    result_dir=result_dir, 
                                    run_again_if_exist=True,
                                    log_period=log_period,
                                    eval_batch_size=eval_batch_size,
                                    suffix_experiment_name=f"{population_size}",
                                    seeds=list(range(0, 5)))
    except Exception as e:
        print("Failed to load:", env_name, e)
        continue

Loading Problems ..:   0%|          | 0/12 [00:00<?, ?it/s]

Successfully loaded: ant



Running ES algorithms:   0%|          | 0/1 [00:00<?, ?it/s]

running the experiment ... [../../results/BraxProblem/ant/Open_ES/0]
Generation 001 | Best fitness (Train): 1044.15 | Best fitness in generation (Train): 1044.15 | Mean fitness (Train): 387.88 | Mean fitness (Test): 783.21


Loading Problems ..:   0%|          | 0/12 [01:01<?, ?it/s]

Generation 002 | Best fitness (Train): 1044.15 | Best fitness in generation (Train): 869.35 | Mean fitness (Train): 497.55 | Mean fitness (Test): 691.86




KeyboardInterrupt

